# Few-shot Universal Dependency Parsing with LLMs (Full CoNLL-U Tabular Evaluation)

Baseline experiment evaluating pretrained LLMs **without weight updates** across $N \in \{0, 3, 5, 10\}$ few-shot demonstrations using full 7-column CoNLL-U-style tabular output (`ID \t FORM \t LEMMA \t UPOS \t XPOS \t FEATS \t HEAD \t DEPREL`) under gold tokenization.

**Metrics (all 9 columns from UDPipe 2.0 Table 5):**
- **Tagging & Lemmatization:** `UPOS`, `XPOS`, `UFeats`, `AllTags` (simultaneous accuracy of UPOS, XPOS, and UFeats), and `Lemmas`
- **Dependency Parsing:** `UAS`, `LAS` (with UD relation subtypes stripped, e.g. `nsubj:pass` $\to$ `nsubj`), `MLAS` (Morphology-Aware LAS: correct head, base dependency relation, UPOS, and UFeats, plus identical UPOS/UFeats on all functional children `aux`, `cop`, `mark`, `det`, `clf`, `case`—excluding punctuation and non-content relations per CoNLL 2018), and `BLEX` (Bi-Lexical dependency score: correct head, base dependency relation, and lemma for content words, following the CoNLL 2018 official evaluation script).

## 1. Setup & Configuration

In [7]:
!pip install -q conllu "transformers>=4.44" accelerate bitsandbytes pandas sentencepiece protobuf

In [8]:
import os, re, json, random, glob, time
from pathlib import Path
from collections import Counter
from tqdm.auto import tqdm
import numpy as np, pandas as pd, torch, conllu

# ---------------- CONFIG ----------------
CFG = dict(
    models = {
        "llama": r"C:\Users\alexa\.cache\huggingface\hub\llama-3.1-8b",
        "llama-instruct": r"C:\Users\alexa\.cache\huggingface\hub\models--meta-llama--Meta-Llama-3-8B-Instruct",
        "aya": r"C:\Users\alexa\.cache\huggingface\hub\aya-expanse-8b",
        "bolmo": r"C:\Users\alexa\.cache\huggingface\hub\models--allenai--Bolmo-7B",
    },
    run_models     = ["llama"],
    treebanks      = ["en_ewt", "de_gsd", "af_afribooms", "fo_oft", "bar_maibaam", "fa_perdt", "sq_staf", "hy_armtdp"],
    shots          = [0, 3],
    seeds          = [0],          # demonstration seeds (0-shot runs once with seed 0)
    n_eval         = 200,            # evaluation sentences per treebank (fixed random subset)
    max_eval_words = 40,
    max_demo_words = 25,
    batch_size     = 4,
    demo_source    = {},           # e.g. {"bar_maibaam": "de_gsd"} for cross-lingual demos
    held_out_demos = 50,           # demo pool carved from test when train/dev are absent
    strip_subtypes = True,
    out_dir        = "results",
)

LEVEL  = dict(en_ewt="HR", en_gum="HR", de_gsd="HR", de_hdt="HR",
              af_afribooms="MR", hy_armtdp="MR", fa_perdt="MR",
              fo_farpahc="LR", fo_oft="LR", bar_maibaam="LR", sq_staf="LR")
WEIGHT = dict(HR=1.0, MR=1.5, LR=2.0)

TREEBANKS = {
    "en_ewt":       ("UD_English-EWT", "en_ewt"),
    "en_gum":       ("UD_English-GUM", "en_gum"),
    "de_gsd":       ("UD_German-GSD", "de_gsd"),
    "de_hdt":       ("UD_German-HDT", "de_hdt"),
    "af_afribooms": ("UD_Afrikaans-AfriBooms", "af_afribooms"),
    "fo_farpahc":   ("UD_Faroese-FarPaHC", "fo_farpahc"),
    "fo_oft":       ("UD_Faroese-OFT", "fo_oft"),
    "bar_maibaam":  ("UD_Bavarian-MaiBaam", "bar_maibaam"),
    "hy_armtdp":    ("UD_Armenian-ArmTDP", "hy_armtdp"),
    "sq_staf":      ("UD_Albanian-STAF", "sq_staf"),
    "fa_perdt":     ("UD_Persian-PerDT", "fa_perdt"),
}

DATA_DIR = "data"
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(CFG["out_dir"], exist_ok=True)

f = Path("HF_TOKEN")
if f.exists():
    os.environ["HF_TOKEN"] = f.read_text().strip()

## 2. Data Loading & Trivial Baseline

In [9]:
def fetch(repo, prefix, split):
    path = f"{DATA_DIR}/{prefix}-ud-{split}.conllu"
    if not (os.path.exists(path) and os.path.getsize(path) > 0):
        url = f"https://raw.githubusercontent.com/UniversalDependencies/{repo}/master/{prefix}-ud-{split}.conllu"
        try:
            urllib.request.urlretrieve(url, path)
        except Exception:
            if os.path.exists(path):
                os.remove(path)
            return None
    return path

def clean(val):
    if val is None or val == "":
        return "_"
    return str(val).replace(" ", "_").replace("\t", "_").replace("\n", "")

def format_feats(feats):
    if not feats:
        return "_"
    if isinstance(feats, str):
        return clean(feats)
    return "|".join(f"{k}={v}" for k, v in sorted(feats.items()))

def read_conllu(path):
    sents = []
    with open(path, encoding="utf8") as f:
        for tl in conllu.parse_incr(f):
            toks = [t for t in tl if isinstance(t["id"], int)]
            if not toks or [t["id"] for t in toks] != list(range(1, len(toks) + 1)):
                continue
            heads = [t["head"] for t in toks]
            if any(h is None for h in heads) or sum(h == 0 for h in heads) != 1:
                continue
            sents.append(dict(
                id=tl.metadata.get("sent_id"),
                forms=[clean(t["form"]) for t in toks],
                lemmas=[clean(t.get("lemma")) for t in toks],
                upos=[clean(t.get("upos") or t.get("upostag")) for t in toks],
                xpos=[clean(t.get("xpos") or t.get("xpostag")) for t in toks],
                feats=[format_feats(t.get("feats")) for t in toks],
                heads=heads,
                deprels=[clean(t["deprel"]) for t in toks],
            ))
    return sents

def available(sents):
    """Which gold columns carry real information (not just '_')."""
    n = sum(len(s["forms"]) for s in sents)
    cov = lambda k: sum(x != "_" for s in sents for x in s[k]) / max(n, 1)
    return dict(lemma=cov("lemmas") > 0.5,
                xpos =cov("xpos")   > 0.5,
                feats=cov("feats")  > 0.05)

DATA = {}

def load_treebank(name):
    repo, prefix = TREEBANKS[name]
    paths = {s: fetch(repo, prefix, s) for s in ["train", "dev", "test"]}
    test = read_conllu(paths["test"])
    if paths["train"]:
        pool, src = read_conllu(paths["train"]), "train"
    elif paths["dev"]:
        pool, src = read_conllu(paths["dev"]), "dev"
    else:
        rng = random.Random(123)
        rng.shuffle(test)
        k = CFG["held_out_demos"]
        pool, test, src = test[:k], test[k:], "test-heldout"
    pool = [s for s in pool if len(s["forms"]) <= CFG["max_demo_words"]]
    ev = [s for s in test if len(s["forms"]) <= CFG["max_eval_words"]]
    random.Random(123).shuffle(ev)
    ev = ev[:CFG["n_eval"]]
    av = available(ev)
    DATA[name] = dict(pool=pool, eval=ev, pool_src=src, avail=av)
    print(f"{name:14s} pool={len(pool):6d} ({src})  eval={len(ev)}  avail={av}")

for tb in set(CFG["treebanks"]) | set(CFG["demo_source"].values()):
    load_treebank(tb)

af_afribooms   pool=   751 (train)  eval=200  avail={'lemma': True, 'xpos': True, 'feats': True}
fo_oft         pool=    50 (test-heldout)  eval=200  avail={'lemma': True, 'xpos': True, 'feats': True}
bar_maibaam    pool=    48 (test-heldout)  eval=200  avail={'lemma': False, 'xpos': False, 'feats': False}
en_ewt         pool= 10190 (train)  eval=200  avail={'lemma': True, 'xpos': True, 'feats': True}
hy_armtdp      pool=  3319 (train)  eval=200  avail={'lemma': True, 'xpos': False, 'feats': True}
fa_perdt       pool= 21727 (train)  eval=200  avail={'lemma': True, 'xpos': True, 'feats': True}
sq_staf        pool=   127 (train)  eval=20  avail={'lemma': True, 'xpos': False, 'feats': True}
de_gsd         pool= 10874 (train)  eval=200  avail={'lemma': True, 'xpos': True, 'feats': True}


## 3. CoNLL-U Tabular Format

Implements all 9 evaluation metrics from UDPipe 2.0 Table 5 (matching the CoNLL 2018 Shared Task `conll18_ud_eval.py` definitions under gold tokenization):
- `UPOS`, `XPOS`, `UFeats`, `AllTags` (joint exact match of UPOS + XPOS + UFeats), `Lemmas`
- `UAS`, `LAS` (base UD relations without subtypes)
- `MLAS` (evaluated on content words—all words whose base deprel is not in `{punct, cc, aux, cop, mark, det, clf, case, goeswith, reparandum, flat, compound, list, orphan, fixed}`—requiring correct head, base deprel, UPOS, and UFeats, plus exact match on the set of `(base_deprel, UPOS, UFeats)` tuples of their functional children `{aux, cop, mark, det, clf, case}`)
- `BLEX` (evaluated on the same content words, requiring correct head, base deprel, and lemma).

In [10]:
METRICS_T5 = ["UPOS", "XPOS", "UFeats", "AllTags", "Lemmas", "UAS", "LAS", "MLAS", "BLEX"]

# CoNLL 2018 functional relations & non-content relations for MLAS and BLEX
FUNC_RELS = {"aux", "cop", "mark", "det", "clf", "case", "cc"}
NON_CONTENT_RELS = FUNC_RELS | {"punct", "goeswith", "reparandum", "cc", "flat", "compound", "list", "orphan", "fixed"}

CONTENT_RELS = {"nsubj","obj","iobj","csubj","ccomp","xcomp","obl","vocative","expl","dislocated",
                "advcl","advmod","discourse","nmod","appos","nummod","acl","amod","conj","fixed",
                "flat","compound","list","parataxis","orphan","goeswith","reparandum","root","dep"}

def base(label):
    return label.split(":")[0]

def norm_feats(f):
    if not f or f == "_":
        return "_"
    parts = sorted(p.strip() for p in f.split("|") if p.strip() and p.strip() != "_")
    return "|".join(parts) if parts else "_"

def table_of(s, strip=True):
    rows = []
    for i in range(len(s["forms"])):
        rel = (base(s["deprels"][i]) if strip else s["deprels"][i]).lower()
        rows.append(
            f"{i+1}\t{s['forms'][i]}\t{s['lemmas'][i]}\t{s['upos'][i]}\t"
            f"{s['xpos'][i]}\t{s['feats'][i]}\t{s['heads'][i]}\t{rel}"
        )
    return "\n".join(rows)

def parse_table(txt, n):
    out = dict(
        lemmas=[None] * n,
        upos=[None] * n,
        xpos=[None] * n,
        feats=[None] * n,
        heads=[None] * n,
        labs=[None] * n,
    )
    for line in txt.split("\n"):
        p = line.split("\t")
        if len(p) < 8:
            p = line.split()
            if len(p) < 8:
                continue
        if not p[0].strip().isdigit():
            continue
        i = int(p[0])
        h = p[6].strip()
        if 1 <= i <= n and out["heads"][i - 1] is None and h.isdigit():
            out["lemmas"][i - 1] = p[2].strip()
            out["upos"][i - 1]   = p[3].strip().upper()
            out["xpos"][i - 1]   = p[4].strip()
            out["feats"][i - 1]  = norm_feats(p[5].strip())
            out["heads"][i - 1]  = int(h)
            out["labs"][i - 1]   = p[7].strip().lower()
    return out

def _func_children(heads, rels, upos, feats):
    n = len(heads)
    fc = [[] for _ in range(n)]
    for j in range(n):
        h = heads[j]
        if h is not None and 1 <= h <= n and rels[j] in FUNC_RELS:
            fc[h - 1].append((rels[j], upos[j], feats[j]))
    return [sorted(x) for x in fc]

def evaluate(s, pred, strip=True):
    n = len(s["forms"])
    p = parse_table(pred, n)
    parsed = sum(h is not None for h in p["heads"])
    f = base if strip else (lambda x: x)

    g_upos  = [u.upper() for u in s["upos"]]
    g_xpos  = s["xpos"]
    g_feats = [norm_feats(ft) for ft in s["feats"]]
    g_lemma = s["lemmas"]
    g_heads = s["heads"]
    g_rels  = [f(d).lower() for d in s["deprels"]]

    p_rels  = [f(r).lower() if r is not None else None for r in p["labs"]]

    g_fc = _func_children(g_heads, g_rels, g_upos, g_feats)
    p_fc = _func_children(p["heads"], p_rels, p["upos"], p["feats"]) if parsed > 0 else [[] for _ in range(n)]

    c_upos = c_xpos = c_feats = c_alltags = c_lemma = c_head = c_las = 0
    n_content = c_mlas = c_blex = 0

   #n_sys_content += int(p_rels[j] in CONTENT_RELS)

    for j in range(n):
        is_content = g_rels[j] not in NON_CONTENT_RELS
        if is_content:
            n_content += 1
        if parsed == 0 or p["heads"][j] is None:
            continue

        u_ok = p["upos"][j] == g_upos[j]
        x_ok = p["xpos"][j] == g_xpos[j]
        f_ok = p["feats"][j] == g_feats[j]
        l_ok = p["lemmas"][j] == g_lemma[j]
        h_ok = p["heads"][j] == g_heads[j]
        las_ok = h_ok and (p_rels[j] == g_rels[j])

        c_upos    += int(u_ok)
        c_xpos    += int(x_ok)
        c_feats   += int(f_ok)
        c_alltags += int(u_ok and x_ok and f_ok)
        c_lemma   += int(l_ok)
        c_head    += int(h_ok)
        c_las     += int(las_ok)

        if is_content and las_ok:
            if u_ok and f_ok and p_fc[j] == g_fc[j]:
                c_mlas += 1
            if l_ok:
                c_blex += 1

    return dict(
        n_gold=n, n_content=n_content, valid=parsed > 0,
        c_upos=c_upos, c_xpos=c_xpos, c_feats=c_feats, c_alltags=c_alltags,
        c_lemma=c_lemma, c_head=c_head, c_las=c_las, c_mlas=c_mlas, c_blex=c_blex,
    )

def summarize(rs, avail=None):
    avail = avail or dict(lemma=True, xpos=True, feats=True)
    g  = sum(r["n_gold"] for r in rs)
    gc = sum(r["n_content"] for r in rs)
    out = dict(
        UPOS    = 100 * sum(r["c_upos"]    for r in rs) / g,
        XPOS    = 100 * sum(r["c_xpos"]    for r in rs) / g,
        UFeats  = 100 * sum(r["c_feats"]   for r in rs) / g,
        AllTags = 100 * sum(r["c_alltags"] for r in rs) / g,
        Lemmas  = 100 * sum(r["c_lemma"]   for r in rs) / g,
        UAS     = 100 * sum(r["c_head"]    for r in rs) / g,
        LAS     = 100 * sum(r["c_las"]     for r in rs) / g,
        MLAS    = 100 * sum(r["c_mlas"]    for r in rs) / max(gc, 1),
        BLEX    = 100 * sum(r["c_blex"]    for r in rs) / max(gc, 1),
    )
    if not avail["xpos"]:
        out["XPOS"] = np.nan
    if not avail["lemma"]:
        out["Lemmas"] = out["BLEX"] = np.nan
    if not avail["feats"]:
        out["UFeats"] = out["MLAS"] = np.nan
    if not (avail["xpos"] and avail["lemma"] and avail["feats"]):
        out["AllTags"] = np.nan
    return out

# Offline oracle verification (all 9 Table 5 metrics must score 100.0)
for tb in CFG["treebanks"]:
    orc = summarize([evaluate(x, table_of(x, CFG["strip_subtypes"]), CFG["strip_subtypes"]) for x in DATA[tb]["eval"]])
    for m_name, val in orc.items():
        assert abs(val - 100.0) < 1e-6, (tb, m_name, val)
print("Oracle check passed (all 9 UDPipe 2.0 Table 5 metrics = 100.0 across all treebanks).")

Oracle check passed (all 9 UDPipe 2.0 Table 5 metrics = 100.0 across all treebanks).


## 4. Prompts, Model Loading & Generation

In [11]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, StoppingCriteria, StoppingCriteriaList

LANG = dict(en="English", de="German", af="Afrikaans", fo="Faroese", bar="Bavarian", hy="Armenian", fa="Persian", sq="Albanian")
def lang_of(tb):
    return LANG.get(tb.split("_")[0], tb.split("_")[0])

UPOS_SET = "ADJ ADP ADV AUX CCONJ DET INTJ NOUN NUM PART PRON PROPN PUNCT SCONJ SYM VERB X"
DEPREL_SET = ("acl advcl advmod amod appos aux case cc ccomp clf compound conj cop csubj dep det "
              "discourse dislocated expl fixed flat goeswith iobj list mark nmod nsubj nummod obj obl "
              "orphan parataxis punct reparandum root vocative xcomp")

# INSTR_TAB = (
#     "Universal Dependencies morphological analysis, lemmatization, and dependency parsing of {lang} sentences. "
#     "For each sentence, write a table in CoNLL-U style, one word per line, with eight tab-separated columns:\n"
#     "ID\tFORM\tLEMMA\tUPOS\tXPOS\tFEATS\tHEAD\tDEPREL\n"
#     "Use '_' when XPOS or FEATS is absent. HEAD is the ID of the word's syntactic head (0 for the root word, "
#     "whose DEPREL is root). DEPREL is a Universal Dependencies relation without subtypes. "
#     "Exactly one word has HEAD 0.\n\n"
# )
INSTR_TAB = (
    "You are an expert Universal Dependencies (UD v2) annotator. Annotate {lang} sentences.\n\n"
    "You receive a table with columns ID and FORM (one token per line, tab-separated). "
    "Return the same table extended to eight tab-separated columns:\n"
    "ID\tFORM\tLEMMA\tUPOS\tXPOS\tFEATS\tHEAD\tDEPREL\n\n"
    "Rules:\n"
    "- Copy ID and FORM exactly. Output exactly one line per input token, in the same order, and nothing else.\n"
    "- LEMMA: dictionary base form.\n"
    f"- UPOS: one of {UPOS_SET}.\n"
    "- XPOS: language-specific tag, following the examples; '_' if absent.\n"
    "- FEATS: UD features as Key=Value joined by '|' in alphabetical order; '_' if none.\n"
    "- HEAD: the ID of the token's syntactic head, an integer between 1 and N (N = number of tokens), "
    "or 0 for the root only. A token is never its own head. Exactly one token has HEAD 0, and its DEPREL is root. "
    "Every token must reach the root (no cycles).\n"
    f"- DEPREL: one of {DEPREL_SET}. Lowercase, no subtypes (nsubj, not nsubj:pass).\n\n"
    "UD attachment conventions (content words are heads):\n"
    "- Function words attach to the content word they modify: case/mark/det/aux/cop/cc attach UP to their content word. "
    "A preposition attaches to its noun (case), not the reverse.\n"
    "- Copula: the predicate (noun/adjective) is the head and the copula is cop, e.g. 'He is tall': tall=root, is=cop.\n"
    "- Coordination: conj attaches to the first conjunct, cc attaches to the following conjunct.\n"
    "- punct attaches to the head of the phrase or clause it closes or separates.\n"
    "- The root is usually the main predicate (verb, or the predicate noun/adjective in copular clauses).\n\n"
)

def build_prompt(demos, s, lang=""):
    p = INSTR_TAB.replace("{lang}", lang)
    for d in demos:
        p += f"Sentence: {' '.join(d['forms'])}\n{table_of(d, strip=CFG['strip_subtypes'])}\n\n"
    return p + f"Sentence: {' '.join(s['forms'])}\n"

def get_demos(tb_eval, N, seed):
    src = CFG["demo_source"].get(tb_eval, tb_eval)
    pool = list(DATA[src]["pool"])
    random.Random(f"{src}-{seed}").shuffle(pool)
    return pool[:N]

MODEL = TOK_ = None

def load_model(hf_id):
    global MODEL, TOK_
    MODEL = TOK_ = None
    import gc; gc.collect(); torch.cuda.empty_cache()

    dt = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16

    tok = AutoTokenizer.from_pretrained(hf_id, trust_remote_code=True)
    tok.padding_side = "left"
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token

    model = AutoModelForCausalLM.from_pretrained(
    hf_id,
    torch_dtype=dt,
    device_map={"": 0},
    attn_implementation="sdpa",
    trust_remote_code=True,
    )
    model.eval()
    MODEL, TOK_ = model, tok
    ctx = getattr(model.config, "max_position_embeddings", 4096)
    print(hf_id, "loaded; context =", ctx)
    return ctx

def fit_demos(demos, s, ctx, reserve, lang=""):
    while True:
        p = build_prompt(demos, s, lang)
        if len(TOK_(p).input_ids) + reserve <= ctx or not demos:
            return demos, p
        demos = demos[:-1]

@torch.no_grad()

class RowStop(StoppingCriteria):
    """Stop each sequence once it has written its n table rows."""
    def __init__(self, tok, n_rows):
        self.tok = tok
        self.n = torch.tensor(n_rows)
        self.cnt = torch.zeros(len(n_rows), dtype=torch.long)
        self.started = torch.zeros(len(n_rows), dtype=torch.bool)
        self.done = torch.zeros(len(n_rows), dtype=torch.bool)
    def __call__(self, input_ids, scores, **kw):
        for i, t in enumerate(self.tok.batch_decode(input_ids[:, -1:])):
            if self.done[i]:
                continue
            if not self.started[i] and t.strip() == "":
                continue            # ignore leading whitespace
            self.started[i] = True
            self.cnt[i] += t.count("\n")
        self.done |= self.cnt >= self.n
        return self.done.to(input_ids.device)

@torch.no_grad()
def generate(prompts, max_new, n_rows):
    enc = TOK_(prompts, return_tensors="pt", padding=True).to(MODEL.device)
    out = MODEL.generate(**enc, max_new_tokens=max_new, do_sample=False,
                         pad_token_id=TOK_.pad_token_id,
                         stopping_criteria=StoppingCriteriaList([RowStop(TOK_, n_rows)]))
    texts = TOK_.batch_decode(out[:, enc.input_ids.shape[1]:], skip_special_tokens=True)
    return [t.lstrip("\n").split("\n\n")[0].split("Sentence:")[0].rstrip() for t in texts]

def run_batch(batch, prompts):
    torch.cuda.empty_cache()
    n_rows = [len(s["forms"]) for s in batch]
    reserve = max(len(TOK_(table_of(s)).input_ids) for s in batch)
    max_new = min(int(reserve * 1.4) + 24, 2048)
    try:
        return generate(prompts, max_new, n_rows)
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache()
        return [generate([p], max_new, [n])[0] for p, n in zip(prompts, n_rows)]


C:\Users\alexa\AppData\Local\Temp\ipykernel_12464\2471338404.py:90: FutureWarning: Decorating classes is deprecated and will be disabled in future versions. You should only decorate functions or methods. To preserve the current behavior of class decoration, you can directly decorate the `__init__` method and nothing else.
  @torch.no_grad()


## 5. Experiment Loop (Resumable)

In [12]:
def run_key(m, tb, N, seed):
    return f"{m}__{tb}__{CFG['demo_source'].get(tb, tb)}__N{N}__s{seed}"

def run_all():
    os.makedirs(CFG["out_dir"], exist_ok=True)
    for m in CFG["run_models"]:
        todo = [(tb, N, sd) for tb in CFG["treebanks"] for N in CFG["shots"]
                for sd in (CFG["seeds"] if N > 0 else [0])
                if not os.path.exists(f"{CFG['out_dir']}/{run_key(m, tb, N, sd)}.json")]
        if not todo:
            print(m, ": all done")
            continue
        ctx = load_model(CFG["models"][m])
        for tb, N, sd in todo:
            t0 = time.time()
            ev = DATA[tb]["eval"]
            demos0 = get_demos(tb, N, sd)
            items = []
            for s in ev:
                reserve = int(len(TOK_(table_of(s)).input_ids) * 1.4) + 24
                d, p = fit_demos(demos0, s, ctx, reserve=reserve, lang=lang_of(tb))
                items.append((s, p))
            gold_lens    = [len(TOK_(table_of(s)).input_ids) for s, _ in items]
            prompt_lens  = [len(TOK_(p).input_ids) for _, p in items]
            max_new_item = [min(int(1.4 * g) + 24, 3072) for g in gold_lens]
            order   = sorted(range(len(items)), key=lambda i: gold_lens[i])
            batches = [order[b:b + CFG["batch_size"]] for b in range(0, len(order), CFG["batch_size"])]
            preds   = [None] * len(items)
            done    = 0
            bar     = tqdm(batches, desc=f"{m} {tb} N={N}", unit="batch", leave=False)
            for idx in bar:
                outs = run_batch([items[i][0] for i in idx], [items[i][1] for i in idx])
                for i, o in zip(idx, outs):
                    preds[i] = o
                done += len(idx)
                bar.set_postfix(sents=f"{done}/{len(order)}", bs=len(idx))
            bar.close()
            res = [evaluate(s, pr, CFG["strip_subtypes"]) for (s, _), pr in zip(items, preds)]
            summ = summarize(res, DATA[tb]["avail"])
            with open(f"{CFG['out_dir']}/{run_key(m, tb, N, sd)}.json", "w") as out_f:
                json.dump(dict(model=m, treebank=tb, N=N, seed=sd, summary=summ,
                               preds=[dict(id=s["id"], pred=pr) for (s, _), pr in zip(items, preds)]), out_f)
            print(f"{m:10s} {tb:13s} N={N:2d} s={sd}  "
                  f"UPOS={summ['UPOS']:5.2f} Lem={summ['Lemmas']:5.2f} "
                  f"UAS={summ['UAS']:5.2f} LAS={summ['LAS']:5.2f} "
                  f"MLAS={summ['MLAS']:5.2f} BLEX={summ['BLEX']:5.2f}  ({time.time()-t0:.0f}s)")

run_all()


llama : all done


## 6. Results Summary

In [13]:
files = sorted(glob.glob(f"{CFG['out_dir']}/*.json"))
if not files:
    print(f"No result files found in {CFG['out_dir']}. Run Section 5 first.")
else:
    rows = []
    valid_rows = []
    for f in files:
        j = json.load(open(f))
        row = dict(model=j["model"], treebank=j["treebank"], N=j["N"], seed=j["seed"])
        row.update({k: j["summary"][k] for k in METRICS_T5})
        
        rows.append(row)

        # Compute invalid (0 valid rows parsed) and partial (<100% tokens parsed) rates
        ev_by_id = {s["id"]: s for s in DATA[j["treebank"]]["eval"]}
        n_sents = len(j["preds"])
        n_invalid = 0
        n_partial = 0
        parsed_tok_ratios = []
        for p in j["preds"]:
            s = ev_by_id[p["id"]]
            parsed = parse_table(p["pred"], len(s["forms"]))
            n_ok = sum(h is not None for h in parsed["heads"])
            if n_ok == 0:
                n_invalid += 1
            elif n_ok < len(s["forms"]):
                n_partial += 1
            parsed_tok_ratios.append(n_ok / len(s["forms"]))

        valid_rows.append({
            "model": j["model"],
            "treebank": j["treebank"],
            "N": j["N"],
            "seed": j["seed"],
            "n_sents": n_sents,
            "invalid (%)": round(100 * n_invalid / n_sents, 2) if n_sents else float("nan"),
            "partial (%)": round(100 * n_partial / n_sents, 2) if n_sents else float("nan"),
            "parsed_toks (%)": round(100 * np.mean(parsed_tok_ratios), 2) if parsed_tok_ratios else float("nan"),
            "UPOS": round(j["summary"]["UPOS"], 2),
            "LAS": round(j["summary"]["LAS"], 2),
        })

    df = pd.DataFrame(rows)
    df_valid = pd.DataFrame(valid_rows)

    # 1. Macro-average across treebanks (averaged over seeds first, matching Table 5 layout)
    tb_mean = df.groupby(["model", "N", "treebank"])[METRICS_T5].mean().reset_index()
    macro = tb_mean.groupby(["model", "N"])[METRICS_T5].mean().round(2)

    print("=== Macro-Average Across Treebanks (UDPipe 2.0 Table 5 Metrics) ===")
    display(macro)

    # 2. Per-model breakdown across individual treebanks (mean over seeds)
    for (m, N), g in tb_mean.groupby(["model", "N"]):
        tab = g.set_index("treebank")[METRICS_T5].copy()
        tab.loc["macro-avg"] = tab.mean()
        print(f"\n=== Model: {m} | N={N} shots ===")
        display(tab.round(2))

    # 3. Parse validity & token coverage diagnostics
    print("\n=== Parse Validity & Token Coverage (Macro-Average by Model & N) ===")
    display(
        df_valid.groupby(["model", "N"])[["invalid (%)", "partial (%)", "parsed_toks (%)", "UPOS", "LAS"]]
        .mean()
        .round(2)
    )

    print("\n=== Parse Validity & Token Coverage (Per Run) ===")
    display(df_valid)

=== Macro-Average Across Treebanks (UDPipe 2.0 Table 5 Metrics) ===


UPOS   XPOS  UFeats  AllTags  Lemmas    UAS    LAS   MLAS  \
model          N                                                               
llama          0  22.00   5.53   11.16     1.62   17.26  11.91   7.57   0.97   
               3  63.02  55.45   48.55    43.65   59.37  29.44  21.74   8.38   
llama-instruct 0   0.00   0.00    0.00     0.00    0.00   0.00   0.00   0.00   
               3  59.81  54.92   51.58    41.48   58.03  31.60  22.34  11.00   

                   BLEX  
model          N         
llama          0   4.30  
               3  15.73  
llama-instruct 0   0.00  
               3  18.67


=== Model: llama | N=0 shots ===


,UPOS,XPOS,UFeats,AllTags,Lemmas,UAS,LAS,MLAS,BLEX
treebank,,,,,,,,,
af_afribooms,20.96,0.00,5.26,0.00,17.16,10.10,6.07,0.04,3.77
bar_maibaam,19.14,NaN,NaN,NaN,NaN,12.45,7.26,NaN,NaN
de_gsd,27.48,8.96,18.77,3.41,20.27,16.40,10.39,2.59,6.35
en_ewt,36.98,17.67,18.38,3.96,24.17,18.65,11.44,1.08,7.33
fa_perdt,31.13,0.25,16.15,0.06,30.27,14.28,10.51,1.07,4.78
fo_oft,32.55,0.75,14.35,0.68,22.55,17.39,12.48,1.44,6.22
hy_armtdp,4.78,NaN,1.95,NaN,3.99,2.74,1.88,0.05,1.12
sq_staf,2.96,NaN,3.23,NaN,2.42,3.23,0.54,0.50,0.50
macro-avg,22.00,5.53,11.16,1.62,17.26,11.91,7.57,0.97,4.30



=== Model: llama | N=3 shots ===


,UPOS,XPOS,UFeats,AllTags,Lemmas,UAS,LAS,MLAS,BLEX
treebank,,,,,,,,,
af_afribooms,51.23,30.88,41.05,30.00,47.95,20.80,13.55,4.77,9.28
bar_maibaam,65.87,NaN,NaN,NaN,NaN,31.10,23.61,NaN,NaN
de_gsd,54.40,45.76,45.47,34.59,50.08,25.07,18.64,8.63,14.55
en_ewt,59.01,55.36,51.49,42.77,57.90,30.71,23.41,13.25,22.75
fa_perdt,77.68,68.78,70.96,60.70,78.50,29.04,22.80,8.27,11.14
fo_oft,74.29,76.46,55.47,50.19,66.34,29.63,24.04,8.21,14.65
hy_armtdp,77.10,NaN,42.33,NaN,71.80,36.91,27.97,9.49,26.17
sq_staf,44.62,NaN,33.06,NaN,43.01,32.26,19.89,6.03,11.56
macro-avg,63.02,55.45,48.55,43.65,59.37,29.44,21.74,8.38,15.73


... (4 outputs left out)


=== Parse Validity & Token Coverage (Macro-Average by Model & N) ===


invalid (%)  partial (%)  parsed_toks (%)   UPOS    LAS
model          N                                                         
llama          0        42.25        57.75            45.16  22.00   7.57
               3         0.00        23.56            97.40  63.02  21.74
llama-instruct 0       100.00         0.00             0.00   0.00   0.00
               3         0.00        26.50            96.40  59.81  22.34


=== Parse Validity & Token Coverage (Per Run) ===


,model,treebank,N,seed,n_sents,invalid (%),partial (%),parsed_toks (%),UPOS,LAS
0,llama-instruct,en_ewt,0,0,200,100.0,0.0,0.00,0.00,0.00
1,llama-instruct,en_ewt,3,0,200,0.0,26.5,96.40,59.81,22.34
2,llama,af_afribooms,0,0,200,44.0,56.0,47.47,20.96,6.07
3,llama,af_afribooms,3,0,200,0.0,49.5,93.02,51.23,13.55
4,llama,bar_maibaam,0,0,200,36.0,64.0,49.14,19.14,7.26
5,llama,bar_maibaam,3,0,200,0.0,17.5,98.46,65.87,23.61
6,llama,de_gsd,0,0,200,25.5,74.5,62.78,27.48,10.39
7,llama,de_gsd,3,0,200,0.0,27.0,97.44,54.40,18.64
8,llama,en_ewt,0,0,200,17.5,82.5,64.27,36.98,11.44
9,llama,en_ewt,3,0,200,0.0,19.0,98.32,59.01,23.41
